# 97 — Match and review nodal stack source positions (SAFE)

This notebook reviews whether a **Geode-linked nodal stack** and a
**nodal-only stack** may represent the same physical source position.

It uses notebook 96's manifests and applies two independent candidate rules:

- source positions must agree within **0.25 m**;
- receiver positions must agree within **0.25 m**.

For each candidate stack pair, the notebook:

1. selects a common waveform component, initially `Z`;
2. matches receivers one-to-one by nearest position;
3. trims traces to their common time interval;
4. optionally filters both traces identically;
5. estimates normalized cross-correlation and lag;
6. writes trace-level and gather-level QC tables;
7. creates review figures for candidate pairs.

This notebook does **not** combine, overwrite, or resave stack waveforms.

## 1. Configuration

In [ ]:
from pathlib import Path
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from obspy import read
from obspy.signal.cross_correlation import correlate, xcorr_max

PROJECT_ROOT = Path('/Volumes/tachyon/LBSSP_DATA')
MANIFEST_ROOT = PROJECT_ROOT / '96_unified_nodal_stack_manifest'
OUT_ROOT = PROJECT_ROOT / '97_nodal_source_match_review'
FIGURE_ROOT = OUT_ROOT / 'figures'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

STACK_MANIFEST_PATH = MANIFEST_ROOT / '96_nodal_stack_manifest.csv'
FILE_MANIFEST_PATH = MANIFEST_ROOT / '96_nodal_stack_file_manifest.csv'

SOURCE_TOLERANCE_M = 0.25
RECEIVER_TOLERANCE_M = 0.25
COMPONENT = 'Z'

# Trace comparison.
MAX_LAG_S = 0.020
FILTER_FREQMIN_HZ = None
FILTER_FREQMAX_HZ = None
MIN_COMMON_DURATION_S = 0.10
MIN_COMMON_RECEIVERS_FOR_GATHER = 3

# Initial review thresholds; these do not automatically merge anything.
TRACE_ACCEPT_CORRELATION = 0.70
GATHER_ACCEPT_MEDIAN_CORRELATION = 0.70
GATHER_ACCEPT_FRACTION = 0.60
GATHER_MAX_MEDIAN_ABS_LAG_S = 0.010

MAKE_REVIEW_FIGURES = True
MAX_FIGURES = None  # e.g. 30; None means all candidates.

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 240)

print('Manifest root:', MANIFEST_ROOT)
print('Output root:', OUT_ROOT)
print('Source tolerance:', SOURCE_TOLERANCE_M, 'm')
print('Receiver tolerance:', RECEIVER_TOLERANCE_M, 'm')
print('Component:', COMPONENT)

## 2. Load and validate notebook-96 manifests

In [ ]:
for path in [STACK_MANIFEST_PATH, FILE_MANIFEST_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f'Missing notebook-96 output: {path}\n'
            'Run notebook 96 first.'
        )

stacks = pd.read_csv(STACK_MANIFEST_PATH, low_memory=False)
files = pd.read_csv(FILE_MANIFEST_PATH, low_memory=False)

for column in ['source_x_m']:
    stacks[column] = pd.to_numeric(stacks[column], errors='coerce')

files['component'] = files['component'].astype(str).str.upper()
files['file_type'] = files['file_type'].astype(str).str.lower()
files['file_exists'] = files['file_exists'].astype(str).str.lower().isin(
    ['true', '1', 'yes']
)

print('Stack rows:', len(stacks))
print('File rows:', len(files))
display(
    stacks.groupby('catalog_branch').size().reset_index(name='n_stacks')
)

## 3. Create source-position candidate pairs

In [ ]:
geode_stacks = stacks.loc[
    stacks.catalog_branch.eq('geode_linked')
    & stacks.source_x_m.notna()
].copy()

nodal_only_stacks = stacks.loc[
    stacks.catalog_branch.eq('nodal_only')
    & stacks.source_x_m.notna()
].copy()

candidate_rows = []

for line in sorted(set(geode_stacks.line.dropna()) | set(nodal_only_stacks.line.dropna())):
    left = geode_stacks.loc[geode_stacks.line.eq(line)]
    right = nodal_only_stacks.loc[nodal_only_stacks.line.eq(line)]

    for geode in left.itertuples(index=False):
        distances = np.abs(right.source_x_m.to_numpy(dtype=float) - float(geode.source_x_m))
        nearby_indices = np.where(distances <= SOURCE_TOLERANCE_M)[0]

        for index in nearby_indices:
            nodal = right.iloc[index]
            candidate_rows.append({
                'candidate_id': (
                    f"{str(geode.stack_id)}__{str(nodal.stack_id)}"
                ),
                'line': line,
                'geode_linked_stack_id': str(geode.stack_id),
                'geode_source_x_m': float(geode.source_x_m),
                'geode_survey': getattr(geode, 'survey', None),
                'geode_file_no': getattr(geode, 'geode_file_no', np.nan),
                'nodal_only_stack_id': str(nodal.stack_id),
                'nodal_only_source_x_m': float(nodal.source_x_m),
                'nodal_only_group_key': nodal.get('nodal_only_group_key', None),
                'source_distance_m': float(distances[index]),
            })

source_candidates = pd.DataFrame(candidate_rows)

if len(source_candidates):
    source_candidates = source_candidates.sort_values(
        ['line', 'geode_source_x_m', 'source_distance_m']
    ).reset_index(drop=True)

print('Candidate stack pairs:', len(source_candidates))
if len(source_candidates):
    display(source_candidates.head(30))
else:
    print('No cross-branch stack pairs fall within the source tolerance.')

## 4. Resolve one MiniSEED waveform path per stack

In [ ]:
def resolve_mseed_path(stack_id, component):
    rows = files.loc[
        files.stack_id.astype(str).eq(str(stack_id))
        & files.component.eq(component)
        & files.file_type.eq('mseed')
        & files.file_exists
    ].copy()

    if rows.empty:
        return None

    if len(rows) > 1:
        rows = rows.sort_values('file_path')

    return str(rows.iloc[0].file_path)


if len(source_candidates):
    source_candidates['geode_mseed_path'] = source_candidates[
        'geode_linked_stack_id'
    ].map(lambda stack_id: resolve_mseed_path(stack_id, COMPONENT))

    source_candidates['nodal_only_mseed_path'] = source_candidates[
        'nodal_only_stack_id'
    ].map(lambda stack_id: resolve_mseed_path(stack_id, COMPONENT))

    source_candidates['waveforms_available'] = (
        source_candidates.geode_mseed_path.notna()
        & source_candidates.nodal_only_mseed_path.notna()
    )

    print(
        'Candidate pairs with both waveform files:',
        int(source_candidates.waveforms_available.sum()),
        '/',
        len(source_candidates),
    )

    unavailable = source_candidates.loc[~source_candidates.waveforms_available]
    if len(unavailable):
        display(unavailable[
            [
                'candidate_id', 'geode_mseed_path',
                'nodal_only_mseed_path'
            ]
        ])

## 5. Waveform and receiver-matching helpers

In [ ]:
def normalize_component(value):
    text = str(value).strip().upper()
    return text[-1] if text and text[-1] in 'ZNE' else text


def receiver_x_m(trace):
    value = pd.to_numeric(
        getattr(trace.stats, 'receiver_x_m', np.nan),
        errors='coerce',
    )
    if pd.notna(value):
        return float(value)

    try:
        return int(str(trace.stats.station)) / 100.0
    except Exception:
        return np.nan


def prepare_stream(path, component):
    stream = read(str(path))
    selected = stream.select(component=f'*{component}')

    if not len(selected):
        selected = type(stream)(
            trace for trace in stream
            if normalize_component(trace.stats.channel) == component
        )

    prepared = []
    for trace in selected:
        x = receiver_x_m(trace)
        if np.isfinite(x):
            prepared.append((x, trace.copy()))

    return sorted(prepared, key=lambda item: item[0])


def one_to_one_receiver_matches(left, right, tolerance_m):
    candidates = []
    for i, (left_x, _) in enumerate(left):
        for j, (right_x, _) in enumerate(right):
            distance = abs(float(left_x) - float(right_x))
            if distance <= tolerance_m:
                candidates.append((distance, i, j))

    candidates.sort()
    used_left = set()
    used_right = set()
    matches = []

    for distance, i, j in candidates:
        if i in used_left or j in used_right:
            continue
        used_left.add(i)
        used_right.add(j)
        matches.append((i, j, distance))

    return matches


def preprocess_trace(trace):
    out = trace.copy()
    out.detrend('demean')
    out.detrend('linear')
    out.taper(max_percentage=0.02, type='cosine')

    if FILTER_FREQMIN_HZ is not None and FILTER_FREQMAX_HZ is not None:
        out.filter(
            'bandpass',
            freqmin=FILTER_FREQMIN_HZ,
            freqmax=FILTER_FREQMAX_HZ,
            corners=4,
            zerophase=True,
        )
    elif FILTER_FREQMIN_HZ is not None:
        out.filter(
            'highpass',
            freq=FILTER_FREQMIN_HZ,
            corners=4,
            zerophase=True,
        )
    elif FILTER_FREQMAX_HZ is not None:
        out.filter(
            'lowpass',
            freq=FILTER_FREQMAX_HZ,
            corners=4,
            zerophase=True,
        )

    return out


def compare_trace_pair(left_trace, right_trace):
    left = preprocess_trace(left_trace)
    right = preprocess_trace(right_trace)

    # Use the lower sampling rate and their common absolute time interval.
    target_rate = min(
        float(left.stats.sampling_rate),
        float(right.stats.sampling_rate),
    )

    if not np.isclose(left.stats.sampling_rate, target_rate):
        left.resample(target_rate)
    if not np.isclose(right.stats.sampling_rate, target_rate):
        right.resample(target_rate)

    common_start = max(left.stats.starttime, right.stats.starttime)
    common_end = min(left.stats.endtime, right.stats.endtime)
    common_duration = float(common_end - common_start)

    if common_duration < MIN_COMMON_DURATION_S:
        return {
            'status': 'insufficient_common_time',
            'common_duration_s': common_duration,
        }

    left.trim(common_start, common_end, nearest_sample=True)
    right.trim(common_start, common_end, nearest_sample=True)

    npts = min(left.stats.npts, right.stats.npts)
    if npts < 3:
        return {
            'status': 'insufficient_samples',
            'common_duration_s': common_duration,
        }

    a = np.asarray(left.data[:npts], dtype=float)
    b = np.asarray(right.data[:npts], dtype=float)

    finite = np.isfinite(a) & np.isfinite(b)
    if finite.sum() < 3:
        return {
            'status': 'nonfinite_data',
            'common_duration_s': common_duration,
        }

    a = a[finite]
    b = b[finite]
    a -= np.mean(a)
    b -= np.mean(b)

    a_std = np.std(a)
    b_std = np.std(b)
    if a_std == 0 or b_std == 0:
        return {
            'status': 'constant_trace',
            'common_duration_s': common_duration,
        }

    max_shift_samples = max(1, int(round(MAX_LAG_S * target_rate)))
    cc = correlate(a / a_std, b / b_std, max_shift_samples, demean=False, normalize='naive')
    shift_samples, corrcoef = xcorr_max(cc, abs_max=False)

    # ObsPy convention: shift applied to the second signal to align with first.
    lag_s = float(shift_samples) / target_rate

    return {
        'status': 'ok',
        'common_duration_s': common_duration,
        'sampling_rate_hz': target_rate,
        'n_samples': int(len(a)),
        'lag_samples': int(shift_samples),
        'lag_s': lag_s,
        'corrcoef': float(corrcoef),
    }

## 6. Compare common receivers for every candidate stack pair

In [ ]:
trace_rows = []
pair_stream_cache = {}

for candidate in source_candidates.itertuples(index=False):
    if not candidate.waveforms_available:
        continue

    try:
        geode_stream = prepare_stream(candidate.geode_mseed_path, COMPONENT)
        nodal_stream = prepare_stream(candidate.nodal_only_mseed_path, COMPONENT)
        matches = one_to_one_receiver_matches(
            geode_stream,
            nodal_stream,
            RECEIVER_TOLERANCE_M,
        )

        pair_stream_cache[candidate.candidate_id] = (
            geode_stream, nodal_stream, matches
        )

        for geode_index, nodal_index, receiver_distance in matches:
            geode_x, geode_trace = geode_stream[geode_index]
            nodal_x, nodal_trace = nodal_stream[nodal_index]

            result = compare_trace_pair(geode_trace, nodal_trace)

            trace_rows.append({
                'candidate_id': candidate.candidate_id,
                'line': candidate.line,
                'geode_linked_stack_id': candidate.geode_linked_stack_id,
                'nodal_only_stack_id': candidate.nodal_only_stack_id,
                'source_distance_m': candidate.source_distance_m,
                'component': COMPONENT,
                'geode_receiver_x_m': geode_x,
                'nodal_only_receiver_x_m': nodal_x,
                'receiver_distance_m': receiver_distance,
                'geode_station': geode_trace.stats.station,
                'nodal_only_station': nodal_trace.stats.station,
                **result,
            })

    except Exception as exc:
        trace_rows.append({
            'candidate_id': candidate.candidate_id,
            'line': candidate.line,
            'geode_linked_stack_id': candidate.geode_linked_stack_id,
            'nodal_only_stack_id': candidate.nodal_only_stack_id,
            'source_distance_m': candidate.source_distance_m,
            'component': COMPONENT,
            'status': 'pair_processing_error',
            'error': repr(exc),
        })

trace_qc = pd.DataFrame(trace_rows)

print('Trace comparison rows:', len(trace_qc))
if len(trace_qc):
    display(
        trace_qc.groupby('status', dropna=False).size()
        .reset_index(name='n_rows')
    )

## 7. Summarize gather-level evidence

In [ ]:
gather_rows = []

for candidate in source_candidates.itertuples(index=False):
    rows = trace_qc.loc[
        trace_qc.candidate_id.eq(candidate.candidate_id)
        & trace_qc.status.eq('ok')
    ].copy()

    if len(rows):
        accepted = rows.corrcoef >= TRACE_ACCEPT_CORRELATION
        n_common = len(rows)
        fraction_accepted = float(accepted.mean())
        median_corr = float(rows.corrcoef.median())
        minimum_corr = float(rows.corrcoef.min())
        median_lag = float(rows.lag_s.median())
        median_abs_lag = float(rows.lag_s.abs().median())
        lag_mad = float(
            np.median(np.abs(rows.lag_s - np.median(rows.lag_s)))
        )
    else:
        n_common = 0
        fraction_accepted = np.nan
        median_corr = np.nan
        minimum_corr = np.nan
        median_lag = np.nan
        median_abs_lag = np.nan
        lag_mad = np.nan

    enough_receivers = n_common >= MIN_COMMON_RECEIVERS_FOR_GATHER
    passes_waveform_thresholds = (
        enough_receivers
        and median_corr >= GATHER_ACCEPT_MEDIAN_CORRELATION
        and fraction_accepted >= GATHER_ACCEPT_FRACTION
        and median_abs_lag <= GATHER_MAX_MEDIAN_ABS_LAG_S
    )

    if not candidate.waveforms_available:
        automatic_status = 'missing_waveform_product'
    elif not enough_receivers:
        automatic_status = 'insufficient_common_receivers'
    elif passes_waveform_thresholds:
        automatic_status = 'waveform_match_supported'
    else:
        automatic_status = 'waveform_match_not_supported'

    gather_rows.append({
        **candidate._asdict(),
        'n_common_receivers': n_common,
        'n_trace_correlations_passing': (
            int((rows.corrcoef >= TRACE_ACCEPT_CORRELATION).sum())
            if len(rows) else 0
        ),
        'fraction_trace_correlations_passing': fraction_accepted,
        'median_trace_corrcoef': median_corr,
        'minimum_trace_corrcoef': minimum_corr,
        'median_lag_s': median_lag,
        'median_abs_lag_s': median_abs_lag,
        'lag_mad_s': lag_mad,
        'automatic_status': automatic_status,
        'manual_review_status': 'not_reviewed',
        'same_physical_shot_decision': 'undecided',
        'review_notes': '',
    })

gather_qc = pd.DataFrame(gather_rows)

if len(gather_qc):
    gather_qc = gather_qc.sort_values(
        ['line', 'geode_source_x_m', 'source_distance_m']
    ).reset_index(drop=True)

print('Gather-level candidates:', len(gather_qc))
if len(gather_qc):
    display(
        gather_qc.groupby('automatic_status', dropna=False)
        .size().reset_index(name='n_candidates')
    )
    display(gather_qc.head(30))

## 8. Create review figures

In [ ]:
def normalized_data(trace):
    data = np.asarray(preprocess_trace(trace).data, dtype=float)
    scale = np.nanmax(np.abs(data))
    return data / scale if np.isfinite(scale) and scale > 0 else data


def safe_name(value):
    return ''.join(
        character if character.isalnum() or character in '-_.' else '_'
        for character in str(value)
    )


figure_paths = []
candidates_for_figures = gather_qc.copy()
if MAX_FIGURES is not None:
    candidates_for_figures = candidates_for_figures.head(MAX_FIGURES)

if MAKE_REVIEW_FIGURES:
    for candidate in candidates_for_figures.itertuples(index=False):
        rows = trace_qc.loc[
            trace_qc.candidate_id.eq(candidate.candidate_id)
            & trace_qc.status.eq('ok')
        ].copy()

        if rows.empty:
            continue

        fig, ax = plt.subplots(figsize=(12, 8))

        offset = 0
        for row in rows.sort_values('geode_receiver_x_m').itertuples(index=False):
            geode_stream, nodal_stream, matches = pair_stream_cache[
                candidate.candidate_id
            ]

            geode_item = min(
                geode_stream,
                key=lambda item: abs(item[0] - row.geode_receiver_x_m),
            )
            nodal_item = min(
                nodal_stream,
                key=lambda item: abs(item[0] - row.nodal_only_receiver_x_m),
            )

            geode_trace = preprocess_trace(geode_item[1])
            nodal_trace = preprocess_trace(nodal_item[1])

            rate = min(
                float(geode_trace.stats.sampling_rate),
                float(nodal_trace.stats.sampling_rate),
            )
            if not np.isclose(geode_trace.stats.sampling_rate, rate):
                geode_trace.resample(rate)
            if not np.isclose(nodal_trace.stats.sampling_rate, rate):
                nodal_trace.resample(rate)

            start = max(geode_trace.stats.starttime, nodal_trace.stats.starttime)
            end = min(geode_trace.stats.endtime, nodal_trace.stats.endtime)
            geode_trace.trim(start, end, nearest_sample=True)
            nodal_trace.trim(start, end, nearest_sample=True)

            npts = min(geode_trace.stats.npts, nodal_trace.stats.npts)
            if npts < 3:
                continue

            a = np.asarray(geode_trace.data[:npts], dtype=float)
            b = np.asarray(nodal_trace.data[:npts], dtype=float)

            a_scale = np.nanmax(np.abs(a))
            b_scale = np.nanmax(np.abs(b))
            if a_scale > 0:
                a = a / a_scale
            if b_scale > 0:
                b = b / b_scale

            times = np.arange(npts) / rate
            ax.plot(times, a + offset, linewidth=0.8)
            ax.plot(times, b + offset, linewidth=0.8, alpha=0.75)
            ax.text(
                times[-1] if len(times) else 0,
                offset,
                f" x={row.geode_receiver_x_m:.2f} m, r={row.corrcoef:.2f}, lag={row.lag_s*1000:.1f} ms",
                va='center',
                fontsize=7,
            )
            offset += 2.5

        ax.set_title(
            f"{candidate.candidate_id}\n"
            f"source separation={candidate.source_distance_m:.3f} m; "
            f"median r={candidate.median_trace_corrcoef:.3f}; "
            f"status={candidate.automatic_status}"
        )
        ax.set_xlabel('Time from common trace start (s)')
        ax.set_ylabel('Normalized traces with vertical offsets')
        ax.grid(True, alpha=0.25)
        fig.tight_layout()

        figure_path = FIGURE_ROOT / f"{safe_name(candidate.candidate_id)}_{COMPONENT}.png"
        fig.savefig(figure_path, dpi=180)
        plt.close(fig)
        figure_paths.append(str(figure_path))

print('Review figures written:', len(figure_paths))

## 9. Export review tables

In [ ]:
OUTPUTS = {
    'source_candidates': OUT_ROOT / '97_source_position_candidates.csv',
    'trace_qc': OUT_ROOT / '97_candidate_trace_correlation_qc.csv',
    'gather_qc': OUT_ROOT / '97_candidate_gather_review.csv',
    'summary': OUT_ROOT / '97_source_match_review_summary.csv',
}

source_candidates.to_csv(OUTPUTS['source_candidates'], index=False)
trace_qc.to_csv(OUTPUTS['trace_qc'], index=False)
gather_qc.to_csv(OUTPUTS['gather_qc'], index=False)

summary = pd.DataFrame([
    ('source_tolerance_m', SOURCE_TOLERANCE_M),
    ('receiver_tolerance_m', RECEIVER_TOLERANCE_M),
    ('component', COMPONENT),
    ('candidate_stack_pairs', len(source_candidates)),
    ('candidate_pairs_with_waveforms', int(
        source_candidates.waveforms_available.sum()
    ) if len(source_candidates) else 0),
    ('trace_comparisons', len(trace_qc)),
    ('successful_trace_comparisons', int(
        trace_qc.status.eq('ok').sum()
    ) if len(trace_qc) else 0),
    ('waveform_match_supported', int(
        gather_qc.automatic_status.eq('waveform_match_supported').sum()
    ) if len(gather_qc) else 0),
    ('waveform_match_not_supported', int(
        gather_qc.automatic_status.eq('waveform_match_not_supported').sum()
    ) if len(gather_qc) else 0),
    ('review_figures', len(figure_paths)),
], columns=['metric', 'value'])

summary.to_csv(OUTPUTS['summary'], index=False)
display(summary)

print('\nWritten:')
for key, path in OUTPUTS.items():
    print(f'  {key:18s} {path}')
print('  figures            ', FIGURE_ROOT)

## 10. Interpretation and next step

`automatic_status` is a screening result, not a final scientific decision.

Review `97_candidate_gather_review.csv` and the corresponding figures. Update:

- `manual_review_status`
- `same_physical_shot_decision`
- `review_notes`

Only candidates confirmed as the same physical source should be considered for
a later integration notebook.

The next integration stage should retain both original stack identifiers and
all receiver-level provenance even when two stacks are accepted as one source.